# Liquidation Dynamics in DeFi and the Role of Transaction Fees — reproducible examples

This notebook reproduces every numerical example of the paper (Examples 1–5, Figures 3, 4, 5, 7, 8, 9).
Run all cells: each figure is displayed and saved to `image/` under the filename used in the manuscript.

**Conventions (Section 3, Algorithm 1, equation (9)).** Pool reserves $(A,B)$, fee $\gamma$ on the input asset
(not added to reserves), $k:=(1-\gamma)(1+\ell)$, oracle price $B/A$, $HF=\theta c\,(B/A)/b$.

* Phase 1 (marginal liquidations): cumulative debt repaid after claiming $x$ is $Bx/(A+xk)$;
  bounds $x_c=c/(1+\ell)$, $x_b=bA/(B-bk)$, $x_{cf}$ from (7); $\pi_{liq}=(k-1)Bx_{liq}/(A+x_{liq}k)$.
* Phase 2 (one final call at reserves $(\bar A,\bar B)$, remaining debt $\bar b=b-Bx_{cf}/(A+x_{cf}k)$):
  a call of size $x$ repays $\bar Bx/\bar A$, so the cap is $x_{\kappa b}=\kappa\bar b\bar A/\bar B$;
  $x^*=\bar A(\sqrt{k}-1)/k$; $x_{last}=\min\{x_c-x_{cf},\,x_{\kappa b},\,x^*\}$.
* Realized liquidation profit (Section 3.2): $\max\{L(CF,1),\,L(1,\kappa)\}$.
* Sandwich (9): sell $\Delta$, liquidate, buy back $\Delta$ at $B_2\Delta/((1-\gamma)(A_2-\Delta))$.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

os.makedirs("image", exist_ok=True)
FIGSIZE = (10, 6)

def save(name):
    plt.tight_layout()
    plt.savefig(os.path.join("image", name), dpi=100)
    plt.show()

## Core functions: Algorithm 1 and the sandwich objective (9)

In [ ]:
def algo1(A, B, g, b, c, th, l, CF, kap):
    """Algorithm 1, L(CF, kappa).
    Returns dict with total profit, pi_liq, pi_last, collateral claimed, and the bounds."""
    k = (1 - g) * (1 + l)
    HF0 = th * B * c / (A * b)
    out = dict(profit=0.0, pi_liq=0.0, pi_last=0.0, x=0.0, x_c=np.nan, x_b=np.nan, x_cf=np.nan,
               x_last=0.0, binds="none")
    if HF0 > CF + 1e-12 or k <= 1:            # step 0
        return out
    x_c = c / (1 + l)                           # step 1
    x_b = b * A / (B - b * k) if B > b * k else np.inf
    Gam = B * k - b * k * k                     # step 2 (sign of Lambda as in eq. (7))
    Lam = CF * (2 * A * b * k - B * A) + th * B * A * (1 + l)
    D = Lam ** 2 + 4 * CF * Gam * (CF * b * A ** 2 - th * B * A * c)
    x_cf = max((Lam + np.sqrt(D)) / (2 * CF * Gam), 0.0) if Gam > 0 else np.inf
    x_liq = min(x_c, x_b, x_cf)                 # step 3
    pi_liq = (k - 1) * B * x_liq / (A + x_liq * k)   # step 4
    pi_last, x_last = 0.0, 0.0
    if x_cf < min(x_c, x_b):                    # step 5
        y_cf = B * x_cf * k / (A + x_cf * k)
        b_bar = b - B * x_cf / (A + x_cf * k)   # cumulative repayment of phase 1
        A_bar, B_bar = A + x_cf * k, B - y_cf
        x_kb = kap * b_bar * A_bar / B_bar      # single-call cap
        x_star = A_bar * (np.sqrt(k) - 1) / k
        x_last = min(x_c - x_cf, x_kb, x_star)
        pi_last = B_bar * x_last * k / (A_bar + x_last * k) - B_bar * x_last / A_bar
    out.update(profit=pi_liq + pi_last, pi_liq=pi_liq, pi_last=pi_last, x=x_liq + x_last,
               x_c=x_c, x_b=x_b, x_cf=x_cf, x_last=x_last,
               binds={x_c: "x_c", x_b: "x_b", x_cf: "x_cf"}[x_liq])
    return out


def realized(A, B, g, b, c, th, l, CF, kap):
    """Realized liquidation profit max{L(CF,1), L(1,kappa)} (Section 3.2)."""
    r1 = algo1(A, B, g, b, c, th, l, CF, 1.0)
    r2 = algo1(A, B, g, b, c, th, l, 1.0, kap)
    return r1 if r1["profit"] >= r2["profit"] else r2


def sandwich(A0, B0, g, b, c, th, l, CF, kap, Delta):
    """Objective (9). Returns (total profit, liquidation component). -inf if the buy-back is infeasible."""
    if Delta <= 0:
        return 0.0, 0.0
    y1 = B0 * Delta * (1 - g) / (A0 + Delta * (1 - g))
    A1 = A0 + Delta * (1 - g); B1 = A0 * B0 / A1
    r = realized(A1, B1, g, b, c, th, l, CF, kap)
    k = (1 - g) * (1 + l)
    A2 = A1 + r["x"] * k; B2 = A1 * B1 / A2
    if A2 <= Delta:
        return -np.inf, r["profit"]
    cost = B2 * Delta / ((1 - g) * (A2 - Delta))
    return y1 + r["profit"] - cost, r["profit"]


def delta_at_hf(hf, A0, B0, g, b, c, th):
    """Attack size that brings the health factor to hf (Remark 4)."""
    return (np.sqrt(th * c * A0 * B0 / (hf * b)) - A0) / (1 - g)

theta, ell = 0.85, 0.05   # used throughout

## Example 1 — Figure 3: liquidation profit vs pool depth

CPMM reserves $A'=1{,}000s$, $B'=2{,}000{,}000s$, fee $\gamma=0.3\%$; position $b=10{,}000$,
$\theta=85\%$, $\ell=5\%$, $CF=95\%$, $\kappa=50\%$, $c=bA_0/(\theta B_0)-0.35$.

In [ ]:
b, gamma, CF, kappa = 10_000.0, 0.003, 0.95, 0.5
A0, B0 = 1_000.0, 2_000_000.0
c = b * A0 / (theta * B0) - 0.35
print(f"c = {c:.4f},  HF0 = {theta * (B0 / A0) * c / b:.4f}")

s_grid = np.linspace(0.2, 5, 2000)
L_cf1 = np.array([algo1(A0 * s, B0 * s, gamma, b, c, theta, ell, CF, 1.0)["profit"] for s in s_grid])
L_1k  = np.array([algo1(A0 * s, B0 * s, gamma, b, c, theta, ell, 1.0, kappa)["profit"] for s in s_grid])

i = np.argmin(L_cf1); j = np.argmax(L_cf1 > L_1k)
print(f"min of L(CF,1) = {L_cf1[i]:.2f} at s = {s_grid[i]:.3f};  L(CF,1) overtakes L(1,kappa) at s = {s_grid[j]:.3f}")
print(f"L(CF,1): {L_cf1[0]:.2f} at s=0.2 -> {L_cf1[-1]:.2f} at s=5;  L(1,kappa): {L_1k[0]:.2f} -> {L_1k[-1]:.2f}")

plt.figure(figsize=FIGSIZE)
plt.plot(s_grid, L_cf1, color="blue", label=r"$L(CF,1)$")
plt.plot(s_grid, L_1k, color="orange", label=r"$L(1,\kappa)$")
plt.xlabel(r"Depth scaling factor $s$ (with $A'=sA_0,\ B'=sB_0$)")
plt.ylabel("Profit")
plt.legend(fontsize=12); plt.grid(True)
save("l_compare.png")

## Example 2 — Figure 4: profit vs DEX price for several initial health factors

$\gamma=0$, liquidity $A_0B_0=K=2\times10^9$ with $A_0=\sqrt{K/p}$, $B_0=\sqrt{Kp}$;
$\theta=85\%$, $\ell=5\%$, $CF=80\%$, $\kappa=50\%$, $b=10{,}000$, $c=HF\cdot bA_0/(\theta B_0)$.
Each curve is **normalized by its own maximum** (raw profits are 250–500 stablecoin); say so in the caption.

In [ ]:
b, gamma, CF, kappa, K = 10_000.0, 0.0, 0.8, 0.5, 2e9
log_p = np.linspace(0.7, 9.9, 1200)

plt.figure(figsize=FIGSIZE)
for HF in [0.50, 0.90, 0.92, 0.94, 0.99]:
    prof = []
    for lp in log_p:
        p = np.exp(lp); A0, B0 = np.sqrt(K / p), np.sqrt(K * p)
        c = HF * b * A0 / (theta * B0)
        prof.append(realized(A0, B0, gamma, b, c, theta, ell, CF, kappa)["profit"])
    prof = np.array(prof)
    print(f"HF = {HF:.2f}: raw profit in [{prof.min():.1f}, {prof.max():.1f}], peak at log p = {log_p[np.argmax(prof)]:.2f}")
    plt.plot(log_p, prof / prof.max(), label=f"HF = {HF:.2f}")
plt.xlabel("log DEX Price, log(B/A)")
plt.ylabel("Profit (normalized to each curve's maximum)")
plt.legend(fontsize=12); plt.grid(True)
save("k_mults_12.png")

## Example 3 — Figure 5: profit vs DEX price for a fixed position

$b=10{,}000$, $c=6$, $\theta=85\%$, $\ell=5\%$, $CF=80\%$, $\kappa=50\%$, $\gamma=0$, $K=2\times10^9$,
$p=HF\cdot b/(\theta c)$. The pink line marks the price at which the binding constraint switches from $x_c$ to $x_{cf}$.

In [ ]:
b, c, gamma, CF, kappa, K = 10_000.0, 6.0, 0.0, 0.8, 0.5, 2e9
p_hf1, p_hf08 = b / (theta * c), 0.8 * b / (theta * c)

def profit_at_price(p):
    A0, B0 = np.sqrt(K / p), np.sqrt(K * p)
    return realized(A0, B0, gamma, b, c, theta, ell, CF, kappa)

def xcf_minus_xc(p):          # under the (1, kappa) strategy, where the switch occurs
    A0, B0 = np.sqrt(K / p), np.sqrt(K * p)
    r = algo1(A0, B0, gamma, b, c, theta, ell, 1.0, kappa)
    return r["x_cf"] - r["x_c"]
p_switch = brentq(xcf_minus_xc, p_hf08 + 1e-6, p_hf1 - 1e-6)
print(f"HF=1 at p = {p_hf1:.2f};  HF=0.8 at p = {p_hf08:.2f};  x_c -> x_cf switch at p = {p_switch:.2f}")

prices = np.linspace(0, 2150, 4001)[1:]
prof = np.array([profit_at_price(p)["profit"] for p in prices])
print(f"peak profit {prof.max():.2f} at p = {prices[np.argmax(prof)]:.2f}")

plt.figure(figsize=FIGSIZE)
plt.axvline(p_hf1, color="red", linestyle="--", label="HF=1")
plt.axvline(p_hf08, color="green", linestyle="--", label="HF=0.8")
plt.axvline(p_switch, color="violet", linestyle="--", label=r"$x_c$ to $x_{cf}$")
plt.plot(prices, prof, color="blue", label="Liquidation Profit")
plt.xlabel("DEX Price (B/A)"); plt.ylabel("Profit")
plt.legend(fontsize=12); plt.grid(True)
save("nofee_nodelta.png")

## Example 4 — Figure 7: sandwich attack without fees

Parameters of Example 3 with $p=2{,}000$ ($A_0=1{,}000$, $B_0=2{,}000{,}000$), so $HF_0=1.02$; $\gamma=0$.
The pink line marks the attack size at which the liquidation switches from $x_{cf}$ to $x_c$.

In [ ]:
b, c, gamma, CF, kappa = 10_000.0, 6.0, 0.0, 0.8, 0.5
A0, B0 = 1_000.0, 2_000_000.0
print(f"HF0 = {theta * (B0 / A0) * c / b:.4f}")
d_hf1 = delta_at_hf(1.0, A0, B0, gamma, b, c, theta)
d_hf08 = delta_at_hf(0.8, A0, B0, gamma, b, c, theta)

def xcf_minus_xc_delta(D):
    A1 = A0 + D * (1 - gamma); B1 = A0 * B0 / A1
    r = algo1(A1, B1, gamma, b, c, theta, ell, 1.0, kappa)
    return r["x_cf"] - r["x_c"]
d_switch = brentq(xcf_minus_xc_delta, d_hf1 + 1e-6, d_hf08 - 1e-6)
print(f"HF=1 at Delta = {d_hf1:.2f};  HF=0.8 at Delta = {d_hf08:.2f};  x_cf -> x_c switch at Delta = {d_switch:.2f}")

deltas = np.linspace(0, 140, 2801)
res = np.array([sandwich(A0, B0, gamma, b, c, theta, ell, CF, kappa, D) for D in deltas])
print(f"total profit at Delta=140: {res[-1,0]:.1f};  liquidation profit peak {res[:,1].max():.1f}")

plt.figure(figsize=FIGSIZE)
plt.plot(deltas, res[:, 0], color="blue", label="Total Profit")
plt.plot(deltas, res[:, 1], color="orange", label="Liquidation Profit")
plt.axvline(d_hf1, color="red", linestyle="--", label="HF=1")
plt.axvline(d_hf08, color="green", linestyle="--", label="HF=0.8")
plt.axvline(d_switch, color="violet", linestyle="--", label=r"$x_{cf}$ to $x_c$")
plt.xlabel("Delta"); plt.ylabel("Profit")
plt.legend(fontsize=12); plt.grid(True)
save("front_nofee.png")

## Example 5 — Figures 8 and 9: sandwich attack with fees

$A_0=10{,}000$, $B_0=28{,}000{,}000$, $\gamma=0.3\%$, $\theta=85\%$, $\ell=5\%$, $CF=80\%$, $\kappa=50\%$,
$b=32{,}000$, $c=20.12$ ($HF_0\approx1.50$).

In [ ]:
b, c, CF, kappa = 32_000.0, 20.12, 0.8, 0.5
A0, B0 = 10_000.0, 28_000_000.0
print(f"HF0 = {theta * (B0 / A0) * c / b:.4f}")

def curve(gamma, d_max=4000, n=4001):
    d1 = delta_at_hf(1.0, A0, B0, gamma, b, c, theta)
    deltas = np.sort(np.concatenate([np.linspace(0, d_max, n), [d1 - 1e-6, d1 + 1e-6]]))
    res = np.array([sandwich(A0, B0, gamma, b, c, theta, ell, CF, kappa, D) for D in deltas])
    return deltas, res[:, 0], res[:, 1]

# ---- Figure 8: gamma = 30 bps
gamma = 0.003
d_hf1 = delta_at_hf(1.0, A0, B0, gamma, b, c, theta)
d_hf08 = delta_at_hf(0.8, A0, B0, gamma, b, c, theta)
deltas, total, liq = curve(gamma)
print(f"gamma = 30 bps: HF=1 at Delta = {d_hf1:.1f}; HF=0.8 at Delta = {d_hf08:.1f}")
print(f"  total profit just before / after the jump: {total[deltas < d_hf1][-1]:.0f} / {total[deltas > d_hf1][0]:.0f}")
m = deltas > 0
print(f"  max total profit over Delta > 0: {total[m].max():.1f}  (negative for every Delta > 0)")

plt.figure(figsize=FIGSIZE)
plt.plot(deltas, total, color="blue", label="Total Profit")
plt.plot(deltas, liq, color="orange", label="Liquidation Profit")
plt.axvline(d_hf1, color="red", linestyle="--", label="HF=1")
plt.axvline(d_hf08, color="green", linestyle="--", label="HF=0.8")
plt.xlabel("Delta"); plt.ylabel("Profit")
plt.legend(fontsize=12); plt.grid(True)
save("fee30_delta.png")

In [ ]:
# ---- critical fee gamma*: smallest fee at which no attack size is profitable
def max_profit(gamma):
    d1 = delta_at_hf(1.0, A0, B0, gamma, b, c, theta)
    Ds = np.linspace(d1 + 1e-6, 4000, 3000)
    return max(sandwich(A0, B0, gamma, b, c, theta, ell, CF, kappa, D)[0] for D in Ds)

gamma_star = brentq(max_profit, 0.0010, 0.0025, xtol=1e-7)
print(f"gamma* = {gamma_star:.5f}  ({gamma_star * 1e4:.2f} bps)")

# ---- Figure 9: total profit for several fee levels
gammas = [0.0010, 0.0015, gamma_star, 0.0019, 0.0022]
colors = ["blue", "orange", "black", "violet", "brown"]

plt.figure(figsize=FIGSIZE)
for g, col in zip(gammas, colors):
    deltas, total, _ = curve(g)
    if g == gamma_star:
        plt.plot(deltas, total, color=col, linewidth=1.7, label=rf"$\gamma^* = {gamma_star:.5f}$")
    else:
        plt.plot(deltas, total, color=col, linewidth=1.4, label=f"γ={g:.4f}")
    print(f"gamma = {g*1e4:5.2f} bps: max total profit = {total[deltas > 0].max():9.1f}")
d_hf1 = delta_at_hf(1.0, A0, B0, gamma_star, b, c, theta)
d_hf08 = delta_at_hf(0.8, A0, B0, gamma_star, b, c, theta)
plt.axvline(d_hf1, color="red", linestyle="--", label=rf"HF=1 ($\gamma^*={gamma_star:.5f}$)")
plt.axvline(d_hf08, color="green", linestyle="--", label=rf"HF=0.8 ($\gamma^*={gamma_star:.5f}$)")
plt.axhline(0, color="black", linestyle=":")
plt.xlabel("Delta"); plt.ylabel("Total Profit")
plt.legend(fontsize=10); plt.grid(True)
save("fee_star.png")

## Numbers quoted in the text

Run the cell below to print every number that appears in the examples' text, for checking after any edit.

In [ ]:
print("Example 1: see printed min / crossing above (non-monotone L(CF,1); L(1,kappa) higher only for shallow pools)")
print("Example 3: x_c -> x_cf switch price           ", f"{p_switch:.2f}")
print("Example 4: HF0 = 1.02; HF=1 at Delta           ", f"{delta_at_hf(1.0, 1000., 2e6, 0.0, 10_000., 6., theta):.2f}")
print("Example 5: HF=1 at Delta (30 bps)              ", f"{delta_at_hf(1.0, A0, B0, 0.003, b, c, theta):.1f}")
print("Example 5: HF=0.8 at Delta (30 bps)            ", f"{delta_at_hf(0.8, A0, B0, 0.003, b, c, theta):.1f}")
print("Example 5: gamma* (bps)                        ", f"{gamma_star * 1e4:.2f}")